# What the vehicle reports about itself -- rekon10

**What this is.** For one flight directory, every self-report the vehicle recorded: the FC's
performance monitor, MCU temperature and voltage, thread stacks, logging buffer, per-port
serial throughput, per-channel MAVLink counters, the F9P's monitor messages and raw
measurements, the FC's errors, events and messages; and for every machine with a session
(coordinator, each campod), collectd's CPU, memory, temperature, disk, network and
per-process counters, the throttle logger and the journal.

It **presents** these. There are no thresholds and no verdicts: the point is that the data
is in one visible place, the same way every flight, so a self-reported problem is seen and
a baseline exists to compare a later flight against.

`docs/flight-data-interpretation.md` has what each FC stream does and does not carry.

In [ ]:
# --- papermill parameters ---
flight_dir = "/home/jovyan/datasets/flights/rekon10/2026-10-27-ffocus"
captures_dir = ""   # "" -> <flight_dir>/captures
out_dir = ""        # "" -> <flight_dir>/derived
journal_max_lines = 200   # per session, in the listing of notable journal lines

In [ ]:
import json, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 400); pd.set_option("display.max_colwidth", 160)

REPO = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "analysis"))
from ardupilot_log import parse_log, SUBSYS_MAP, EV_MAP
import vehicle_health as vh

FLIGHT = Path(flight_dir)
CAP = Path(captures_dir) if captures_dir else FLIGHT / "captures"
OUT = Path(out_dir) if out_dir else FLIGHT / "derived"
OUT.mkdir(parents=True, exist_ok=True)

def is_dataflash(p):
    with open(p, "rb") as f: return f.read(3) == b"\xa3\x95\x80"
BINS = sorted(p for p in list(FLIGHT.glob("*.bin")) + list(FLIGHT.glob("fc/*.bin")) if is_dataflash(p))
SESSIONS = sorted(p for p in CAP.glob("*/*") if p.is_dir() and not p.name.startswith("@"))
REPORT = dict(notebook="vehicle-health.ipynb", flight_dir=str(FLIGHT), fc={}, nodes={})

def figsave(fig, name):
    fig.tight_layout(); fig.savefig(OUT / f"vehicle-health-{name.replace(".bin", "")}.png", dpi=100); plt.show(); plt.close(fig)

## 1. What this flight recorded

Absence is a result: a machine with no collectd, a log with no `PM`, a session with no
journal is listed here so it is not mistaken for a quiet one.

In [ ]:
inv = []
for b in BINS:
    inv.append(dict(machine="fc", item=b.name, bytes=b.stat().st_size))
for s in SESSIONS:
    node = s.parent.name
    cd = s / "collectd" / node
    plugins = sorted(p.name for p in cd.glob("*") if p.is_dir()) if cd.is_dir() else []
    inv.append(dict(machine=node, item=f"session {s.name}",
                    bytes=sum(f.stat().st_size for f in s.rglob("*") if f.is_file()),
                    journal=(s / "journal.log").is_file(), collectd_plugins=len(plugins)))
display(pd.DataFrame(inv))
NOT_RECORDED = [
    "coordinator / campod wifi signal (collectd has interface counters, no wireless plugin)",
    "ELRS radio 2 RSSI, downlink, transmit power (not in RADIO_STATUS)",
    "container restart counts as a series (only what the journal says about them)",
    "FC board temperature other than the MCU die (MCU.MTemp)",
]
REPORT["not_recorded"] = NOT_RECORDED
display(Markdown("**Not recorded on this vehicle:**\n\n" + "\n".join(f"- {x}" for x in NOT_RECORDED)))

## 2. The flight controller

One block per DataFlash log. Armed spans (from `EV`) are shaded on every time plot.

In [ ]:
WANT = {"PM", "MCU", "POWR", "STAK", "DSF", "UART", "MAV", "UBX1", "UBX2", "GRXH", "GRXS",
        "TSYN", "ERR", "EV", "MSG", "GPS"}
FC = {}
for b in BINS:
    F, duration, parms = parse_log(b, WANT)
    t0 = min(df["TimeUS"].min() for df in F.values())
    for df in F.values(): df["t"] = (df["TimeUS"] - t0) / 1e6
    armed = []; ev = F.get("EV", pd.DataFrame()); ta = None
    for _, r in ev.sort_values("t").iterrows() if not ev.empty else []:
        if r.Id == 10 and ta is None: ta = r.t
        elif r.Id == 11 and ta is not None: armed.append((ta, r.t)); ta = None
    if ta is not None: armed.append((ta, duration))
    FC[b.name] = dict(F=F, duration=duration, armed=armed, parms=parms, fit=vh.fc_utc_fit(F.get("GPS")))
    REPORT["fc"][b.name] = dict(duration_s=duration, armed_s=armed,
                                messages={k: len(v) for k, v in sorted(F.items())})
    print(f"{b.name}: {duration:.0f} s, armed {[(round(a), round(z)) for a, z in armed]};",
          ", ".join(f"{k} {len(v)}" for k, v in sorted(F.items())))

def shade(ax, armed):
    for a, z in armed: ax.axvspan(a, z, color="tab:orange", alpha=0.12, lw=0)

def series_panel(name, panels, armed, title):
    # panels: list of (label, [(x, y, legend), ...])
    fig, axes = plt.subplots(len(panels), 1, figsize=(12, 1.7 * len(panels) + 0.6), sharex=True)
    for ax, (label, lines) in zip(np.atleast_1d(axes), panels):
        for x, y, leg in lines: ax.plot(x, y, ".-", ms=2, lw=0.6, label=leg)
        shade(ax, armed); ax.set_ylabel(label, fontsize=8); ax.tick_params(labelsize=7)
        if len(lines) > 1: ax.legend(fontsize=6, ncol=6, loc="upper right")
    np.atleast_1d(axes)[-1].set_xlabel("s from log start")
    fig.suptitle(title, fontsize=10); figsave(fig, name)

### 2a. Performance monitor, MCU, power

`PM` (every 10 s or so): `Load` is main-loop load in 0.1 %, `MaxT` the longest loop in us,
`NLon` loops over time out of `NL`, `Mem` free heap bytes, `InE` internal-error bitmask,
`ErC` internal-error count, `ErrL` the line of the last one. `SPIC`/`I2CC` are SPI/I2C
transaction counts and `I2CI` I2C interrupts. `MCU`: die temperature and the 3.3 V rail's
voltage, with its min and max since the previous sample. `POWR`: board 5 V and servo rail
(NaN where the board has no sense pin), and the power flag bitmasks.

In [ ]:
for name, d in FC.items():
    F, armed = d["F"], d["armed"]; rep = REPORT["fc"][name]
    pm = F.get("PM"); mcu = F.get("MCU"); pw = F.get("POWR")
    panels = []
    if pm is not None:
        panels += [("load %", [(pm.t, pm.Load / 10, "Load")]), ("max loop us", [(pm.t, pm.MaxT, "MaxT")]),
                   ("slow loops", [(pm.t, pm.NLon, "NLon")]), ("free heap B", [(pm.t, pm.Mem, "Mem")]),
                   ("internal errors", [(pm.t, pm.ErC, "ErC"), (pm.t, pm.InE, "InE")])]
        rep["PM"] = {c: vh.summarize(pm[c]) for c in ("Load", "MaxT", "NLon", "NL", "Mem", "InE", "ErC", "ErrL")}
        rep["PM"]["InE_bits_seen"] = int(np.bitwise_or.reduce(pm.InE.astype(int).values))
    if mcu is not None:
        panels += [("MCU temp C", [(mcu.t, mcu.MTemp, "MTemp")]),
                   ("MCU V", [(mcu.t, mcu.MVolt, "MVolt"), (mcu.t, mcu.MVmin, "MVmin"), (mcu.t, mcu.MVmax, "MVmax")])]
        rep["MCU"] = {c: vh.summarize(mcu[c]) for c in ("MTemp", "MVolt", "MVmin", "MVmax")}
    if pw is not None:
        rep["POWR"] = {c: vh.summarize(pw[c]) for c in ("Vcc", "VServo")}
        rep["POWR"]["Flags_seen"] = sorted(int(x) for x in pw.Flags.unique())
        rep["POWR"]["AccFlags_seen"] = sorted(int(x) for x in pw.AccFlags.unique())
        print(f"{name} POWR: Vcc {rep['POWR']['Vcc']}, VServo {rep['POWR']['VServo']}, "
              f"Flags seen {rep['POWR']['Flags_seen']}, AccFlags seen {rep['POWR']['AccFlags_seen']}")
    if pm is not None:
        display(pm[["t", "Load", "MaxT", "NLon", "NL", "Mem", "InE", "ErC", "ErrL", "SPIC", "I2CC", "I2CI"]]
                .round(1).set_index("t"))
    if panels: series_panel(f"fc-pm-{name}", panels, armed, f"{name}: performance monitor and MCU")

### 2b. Thread stacks and the logging buffer

`STAK`: per ChibiOS thread, stack size and the least free stack seen (bytes). `DSF`:
DataFlash backend -- `Dp` blocks dropped (cumulative), `FMn` the least free buffer space in
the last interval, `Bytes` written.

In [ ]:
for name, d in FC.items():
    F, armed = d["F"], d["armed"]; rep = REPORT["fc"][name]
    st = F.get("STAK")
    if st is not None:
        tab = st.groupby("Name").agg(total=("Total", "max"), min_free=("Free", "min"),
                                     last_free=("Free", "last"), priority=("Pri", "last"))
        tab["min_free_pct"] = (100 * tab.min_free / tab.total).round(1)
        rep["STAK"] = tab.reset_index().to_dict(orient="records")
        display(tab.sort_values("min_free_pct"))
    ds = F.get("DSF")
    if ds is not None:
        rep["DSF"] = dict(dropped_total=int(ds.Dp.max()), buffer_free_min=vh.summarize(ds.FMn),
                          bytes_written=int(ds.Bytes.max()))
        rr = vh.counter_rate(ds, "Bytes")
        series_panel(f"fc-dsf-{name}", [("dropped blocks", [(ds.t, ds.Dp, "Dp")]),
                                        ("min free buffer B", [(ds.t, ds.FMn, "FMn")]),
                                        ("write B/s", [(rr.t, rr.rate, "Bytes")])], armed,
                     f"{name}: DataFlash logging")

### 2c. Serial ports and MAVLink channels

`UART`: bytes/s transmitted and received per `SERIALn` (`I` is the SERIAL index), and
received bytes dropped. A port absent here moved no bytes. `MAV`: per MAVLink channel
(`chan` = `MAVn` - 1: 0 USB, 1 the coordinator, 2 the ELRS link), packets/s sent and
received, receive drops, and the counters `tf` (times the send buffer was full) and `mgs`
(max gap).

In [ ]:
for name, d in FC.items():
    F, armed = d["F"], d["armed"]; rep = REPORT["fc"][name]
    ua = F.get("UART")
    if ua is not None:
        tab = ua.groupby("I").agg(tx_median=("Tx", "median"), tx_max=("Tx", "max"), rx_median=("Rx", "median"),
                                  rx_max=("Rx", "max"), rx_dropped_total=("RxDp", "sum"), n=("Tx", "size"))
        tab.index = [f"SERIAL{i}" for i in tab.index]
        rep["UART"] = tab.round(1).reset_index(names="port").to_dict(orient="records")
        display(tab.round(1))
        series_panel(f"fc-uart-{name}", [
            ("tx B/s", [(g.t, g.Tx, f"SERIAL{i}") for i, g in ua.groupby("I")]),
            ("rx B/s", [(g.t, g.Rx, f"SERIAL{i}") for i, g in ua.groupby("I")]),
            ("rx dropped B/s", [(g.t, g.RxDp, f"SERIAL{i}") for i, g in ua.groupby("I")])],
            armed, f"{name}: serial ports")
    mv = F.get("MAV")
    if mv is not None:
        rates = {c: vh.counter_rate(mv, c, by="chan") for c in ("txp", "rxp", "rxdp")}
        rep["MAV"] = {}
        for ch, g in mv.groupby("chan"):
            rep["MAV"][f"chan{ch}"] = dict(
                tx_pps=vh.summarize(rates["txp"][rates["txp"].chan == ch].rate),
                rx_pps=vh.summarize(rates["rxp"][rates["rxp"].chan == ch].rate),
                rx_dropped_total=int(g.rxdp.max() - g.rxdp.min()), tf_max=int(g.tf.max()), mgs_max=int(g.mgs.max()),
                flags_seen=sorted(int(x) for x in g["flags"].unique()))
        display(pd.DataFrame({k: {"tx pps median": v["tx_pps"].get("median"), "rx pps median": v["rx_pps"].get("median"),
                                  "rx dropped": v["rx_dropped_total"], "tf max": v["tf_max"], "mgs max": v["mgs_max"],
                                  "flags": v["flags_seen"]} for k, v in rep["MAV"].items()}).T)
        series_panel(f"fc-mav-{name}", [
            ("tx pkt/s", [(g.t, g.rate, f"chan{c}") for c, g in rates["txp"].groupby("chan")]),
            ("rx pkt/s", [(g.t, g.rate, f"chan{c}") for c, g in rates["rxp"].groupby("chan")]),
            ("rx dropped/s", [(g.t, g.rate, f"chan{c}") for c, g in rates["rxdp"].groupby("chan")])],
            armed, f"{name}: MAVLink channels")

### 2d. The F9P's own monitoring

`UBX1`/`UBX2` are UBX-MON-HW as the driver logs it: `noisePerMS`, `jamInd` (0-255 CW
interference), `agcCnt` (0-8191), `aPower` (antenna power: 0 off, 1 on, 2 unknown), and the
I/Q imbalance in `UBX2`. `GRXH`/`GRXS` are RXM-RAWX (`GPS_RAW_DATA`): measurements per epoch,
receiver status, and per satellite carrier-to-noise. `gnss` 0 GPS, 1 SBAS, 2 Galileo, 3 BeiDou,
5 QZSS, 6 GLONASS.

In [ ]:
GNSS = {0: "GPS", 1: "SBAS", 2: "Galileo", 3: "BeiDou", 5: "QZSS", 6: "GLONASS"}
for name, d in FC.items():
    F, armed = d["F"], d["armed"]; rep = REPORT["fc"][name]
    u1, u2, gh, gs = (F.get(k) for k in ("UBX1", "UBX2", "GRXH", "GRXS"))
    panels = []
    if u1 is not None:
        panels += [("noisePerMS", [(u1.t, u1.noisePerMS, "")]), ("jamInd", [(u1.t, u1.jamInd, "")]),
                   ("agcCnt", [(u1.t, u1.agcCnt, "")]), ("aPower", [(u1.t, u1.aPower, "")])]
        rep["UBX1"] = {c: vh.summarize(u1[c]) for c in ("noisePerMS", "jamInd", "agcCnt", "aPower")}
    if u2 is not None:
        panels += [("I/Q mag", [(u2.t, u2.magI, "magI"), (u2.t, u2.magQ, "magQ")])]
        rep["UBX2"] = {c: vh.summarize(u2[c]) for c in ("ofsI", "magI", "ofsQ", "magQ")}
    if gh is not None:
        panels += [("RAWX meas/epoch", [(gh.t, gh.numMeas, "")])]
        rep["GRXH"] = dict(numMeas=vh.summarize(gh.numMeas), recStat_seen=sorted(int(x) for x in gh.recStat.unique()))
    if panels: series_panel(f"fc-gnss-{name}", panels, armed, f"{name}: F9P monitor and raw measurements")
    if gs is not None:
        tab = gs.groupby("gnss").agg(satellites=("sv", "nunique"), samples=("cno", "size"), cno_median=("cno", "median"),
                                     cno_p10=("cno", lambda s: s.quantile(0.1)), cno_max=("cno", "max"))
        tab.index = [GNSS.get(i, str(i)) for i in tab.index]
        rep["GRXS"] = tab.reset_index(names="gnss").to_dict(orient="records")
        display(tab)

### 2e. Errors, events, messages, time sync

Every `ERR` (subsystem, code), `EV` and `MSG` the FC logged, decoded where the code is in
`ardupilot_log`, and the `TSYN` round-trip times.

In [ ]:
for name, d in FC.items():
    F = d["F"]; rep = REPORT["fc"][name]
    er = F.get("ERR")
    if er is not None:
        er = er.assign(subsystem=er.Subsys.map(lambda s: SUBSYS_MAP.get(int(s), str(int(s)))))
        rep["ERR"] = er[["t", "Subsys", "subsystem", "ECode"]].round(3).to_dict(orient="records")
        display(Markdown(f"**{name} ERR**")); display(er[["t", "Subsys", "subsystem", "ECode"]].round(2))
    ev = F.get("EV")
    if ev is not None:
        ev = ev.assign(event=ev.Id.map(lambda s: EV_MAP.get(int(s), str(int(s)))))
        rep["EV"] = ev[["t", "Id", "event"]].round(3).to_dict(orient="records")
        display(Markdown(f"**{name} EV**")); display(ev[["t", "Id", "event"]].round(2))
    ms = F.get("MSG")
    if ms is not None:
        rep["MSG"] = ms[["t", "Message"]].round(3).to_dict(orient="records")
        display(Markdown(f"**{name} MSG**")); display(ms[["t", "Message"]].round(2))
    ts = F.get("TSYN")
    if ts is not None:
        rep["TSYN"] = dict(rtt_us=vh.summarize(ts.RTT), sysids=sorted(int(x) for x in ts.SysID.unique()))
        print(f"{name} TSYN: RTT us {rep['TSYN']['rtt_us']}, peers {rep['TSYN']['sysids']}")

## 3. The machines: collectd, throttling, journal

One block per session. collectd counters are differenced into rates; CPU is each state's
share of all jiffies per core. Where an FC log's GPS fit covers the session, its armed spans
are shaded (on unix time).

In [ ]:
ARMED_UTC = []
for d in FC.values():
    if d["fit"] is None: continue
    s, i = d["fit"]; F = d["F"]; t0us = min(df["TimeUS"].min() for df in F.values())
    for a, z in d["armed"]:
        ARMED_UTC.append(((t0us + a * 1e6) * s + i, (t0us + z * 1e6) * s + i))

def pick(s, plugin, typ, col="value"):
    x = s[(s.plugin == plugin) & (s.type == typ) & (s.column == col)]
    return x[["t", "value"]].sort_values("t")

def rate(s, plugin, typ, col="value"):
    x = pick(s, plugin, typ, col)
    return vh.counter_rate(x, "value") if len(x) > 1 else pd.DataFrame(columns=["t", "rate"])

def node_panels(node, sess, s):
    t0 = s.t.min()
    rel = lambda t: np.asarray(t, float) - t0
    panels = []; rep = {}
    # CPU share per core
    cpu_lines = []; cpus = sorted(p for p in s.plugin.unique() if re.fullmatch(r"cpu-\d+", p))
    busy_all = []
    for p in cpus:
        states = {typ.replace("cpu-", ""): rate(s, p, typ) for typ in s[s.plugin == p].type.unique()}
        if not states: continue
        df = pd.concat({k: v.set_index("t").rate for k, v in states.items()}, axis=1).dropna()
        tot = df.sum(axis=1)
        busy = 100 * (1 - df.get("idle", 0) / tot)
        cpu_lines.append((rel(df.index), busy, p)); busy_all.append(busy)
        if "wait" in df: rep[f"{p}_iowait_pct"] = vh.summarize(100 * df["wait"] / tot)
    if cpu_lines:
        panels.append(("CPU busy %", cpu_lines)); rep["cpu_busy_pct"] = vh.summarize(pd.concat(busy_all))
    mem = [(rel(x.t), x.value / 2**20, typ.replace("memory-", "")) for typ in ("memory-used", "memory-free", "memory-cached")
           for x in [pick(s, "memory", typ)] if len(x)]
    if mem:
        panels.append(("memory MiB", mem))
        rep["memory_free_mib"] = vh.summarize(pick(s, "memory", "memory-free").value / 2**20)
        rep["memory_used_mib"] = vh.summarize(pick(s, "memory", "memory-used").value / 2**20)
    th = [(rel(x.t), x.value, p.replace("thermal-", "")) for p in s.plugin.unique() if p.startswith("thermal-")
          for x in [pick(s, p, "temperature")] if len(x)]
    if th:
        panels.append(("temp C", th))
        rep["temperature_c"] = {l: vh.summarize(y) for _, y, l in th}
    for p in sorted(x for x in s.plugin.unique() if x.startswith("disk-")):
        r_ = rate(s, p, "disk_octets", "write"); io = rate(s, p, "disk_io_time", "io_time")
        if len(r_): panels.append((f"{p} write KiB/s", [(rel(r_.t), r_.rate / 1024, "")]))
        if len(io): panels.append((f"{p} busy ms/s", [(rel(io.t), io.rate, "")]))
        rep[p] = dict(write_kib_s=vh.summarize(r_.rate / 1024) if len(r_) else {},
                      io_time_ms_per_s=vh.summarize(io.rate) if len(io) else {})
    for p in sorted(x for x in s.plugin.unique() if x.startswith("interface-")):
        lines = []; rr = {}
        for typ, col, lab, k in (("if_octets", "rx", "rx KiB/s", 1024), ("if_octets", "tx", "tx KiB/s", 1024)):
            x = rate(s, p, typ, col)
            if len(x): lines.append((rel(x.t), x.rate / k, lab)); rr[lab] = vh.summarize(x.rate / k)
        bad = []
        for typ in ("if_errors", "if_dropped"):
            for col in ("rx", "tx"):
                x = pick(s, p, typ, col)
                if len(x): rr[f"{typ}_{col}_total"] = float(x.value.max() - x.value.min())
                xr = rate(s, p, typ, col)
                if len(xr): bad.append((rel(xr.t), xr.rate, f"{typ} {col}"))
        if lines: panels.append((p.replace("interface-", "") + " KiB/s", lines))
        if bad: panels.append((p.replace("interface-", "") + " err+drop/s", bad))
        rep[p] = rr
    procs = sorted(x for x in s.plugin.unique() if x.startswith("processes-"))
    rss = [(rel(x.t), x.value / 2**20, p.replace("processes-", "")) for p in procs for x in [pick(s, p, "ps_rss")] if len(x)]
    if rss: panels.append(("process RSS MiB", rss))
    cput = []; majf = []; cnt = []
    for p in procs:
        x = rate(s, p, "ps_cputime", "user"); y = rate(s, p, "ps_cputime", "syst")
        if len(x) and len(y):
            j = pd.merge_asof(x, y, on="t", suffixes=("_u", "_s"))
            cput.append((rel(j.t), (j.rate_u + j.rate_s) / 1e4, p.replace("processes-", "")))  # us/s -> % of a core
        f = rate(s, p, "ps_pagefaults", "majflt")
        if len(f): majf.append((rel(f.t), f.rate, p.replace("processes-", "")))
        c = pick(s, p, "ps_count", "processes")
        if len(c): cnt.append((rel(c.t), c.value, p.replace("processes-", "")))
    if cput: panels.append(("process CPU % of a core", cput))
    if majf: panels.append(("major faults/s", majf))
    if cnt: panels.append(("process count", cnt))
    rep["processes"] = {p.replace("processes-", ""): dict(
        rss_mib=vh.summarize(pick(s, p, "ps_rss").value / 2**20),
        count=vh.summarize(pick(s, p, "ps_count", "processes").value),
        major_faults_total=float(np.nansum(np.diff(pick(s, p, "ps_pagefaults", "majflt").value))) if len(pick(s, p, "ps_pagefaults", "majflt")) > 1 else None)
        for p in procs}
    st = [(rel(x.t), x.value, typ.replace("ps_state-", "")) for typ in ("ps_state-blocked", "ps_state-zombies", "ps_state-running")
          for x in [pick(s, "processes", typ)] if len(x)]
    if st: panels.append(("process states", st))
    for typ, lab in (("vmpage_faults", "majflt"),):
        x = rate(s, "vmem", typ, lab)
        if len(x): panels.append(("system major faults/s", [(rel(x.t), x.rate, "")])); rep["system_major_faults_per_s"] = vh.summarize(x.rate)
    x = rate(s, "vmem-throttle", "vmpage_action-scan_direct")
    if len(x): panels.append(("direct reclaim scans/s", [(rel(x.t), x.rate, "")])); rep["direct_reclaim_scans_per_s"] = vh.summarize(x.rate)
    return t0, panels, rep

NOTABLE = re.compile(r"oom|out of memory|killed process|segfault|core dump|panic|oops|"
                     r"fail|error|exited|restart|timed? ?out|under-?volt|throttl|"
                     r"watchdog.*(expir|timeout|reset)", re.I)
QUIET = re.compile(r"pi-throttle\.service|coord-throttle-log|Starting pi-throttle|Finished pi-throttle")

for sess in SESSIONS:
    node = sess.parent.name
    key = f"{node}/{sess.name}"
    display(Markdown(f"### {key}"))
    rep = REPORT["nodes"][key] = {}
    cd = sess / "collectd" / node
    if cd.is_dir():
        s = vh.collectd_series(cd)
        if len(s):
            t0, panels, rep["collectd"] = node_panels(node, sess, s)
            fig, axes = plt.subplots(len(panels), 1, figsize=(12, 1.6 * len(panels) + 0.6), sharex=True)
            for ax, (label, lines) in zip(np.atleast_1d(axes), panels):
                for x, y, leg in lines: ax.plot(x, y, ".-", ms=1.5, lw=0.6, label=leg)
                for a, z in ARMED_UTC: ax.axvspan(a - t0, z - t0, color="tab:orange", alpha=0.12, lw=0)
                ax.set_ylabel(label, fontsize=7); ax.tick_params(labelsize=7)
                if len(lines) > 1: ax.legend(fontsize=6, ncol=6, loc="upper right")
            np.atleast_1d(axes)[-1].set_xlabel("s from the session's first collectd sample")
            fig.suptitle(f"{key}: collectd", fontsize=10); figsave(fig, f"{node}-{sess.name[:8]}")
    else:
        print("no collectd in this session")
    jf = sess / "journal.log"
    if jf.is_file():
        j = vh.journal_lines(jf)
        th = vh.throttle_states(j)
        rep["throttle"] = dict(samples=len(th), values_seen=sorted(th.raw.unique().tolist()),
                               flags_seen=sorted(th["flags"].unique().tolist()))
        print(f"journal: {len(j)} lines; throttle logger: {len(th)} samples, values seen {rep['throttle']['values_seen']}, "
              f"flags {rep['throttle']['flags_seen']}")
        n = j[j.msg.str.contains(NOTABLE, na=False) & ~(j.unit + " " + j.msg).str.contains(QUIET, na=False)]
        rep["journal"] = dict(lines=len(j), notable=len(n),
                              notable_by_unit=n.unit.value_counts().to_dict())
        display(n.unit.value_counts().rename("notable lines").to_frame())
        shown = n.head(journal_max_lines)
        display(Markdown(f"Notable lines ({len(n)}; first {len(shown)} shown) -- matched on words like fail, error, "
                         f"restart, oom, throttl, not graded:"))
        display(shown.assign(t=pd.to_datetime(shown.t, unit="s", utc=True).dt.strftime("%H:%M:%S.%f").str[:-3])
                [["t", "unit", "msg"]].reset_index(drop=True))
    else:
        print("no journal in this session")

## 4. Written out

`derived/vehicle-health.json` holds every figure above, per log and per session, for
comparison with later flights. Figures are `derived/vehicle-health-*.png`.

In [ ]:
(OUT / "vehicle-health.json").write_text(json.dumps(REPORT, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n")
print("wrote", OUT / "vehicle-health.json")